In [4]:
! python -m pip install requests pandas beautifulsoup4

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.2-py3-none-any.whl.metadata (6.6 kB)
  Using cached certifi-2025.11.12-py3-none-any.whl.metadata (2.5 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached soupsieve-2.8.1-py3-none-any.whl.metadata (4.6 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB 960.0 kB/s eta 0:00:12
    --------------------------------------- 0.1/11.3 MB 2.1 MB/s eta 0:00:06
    --------------------------------------- 0.2/11.3 MB 2.1 MB/s eta 0:00:06
    --------------------------------------- 0.2/11.3 MB 1.5 MB/s eta 0:00:08
   - -------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import sys
print(sys.version)
print(sys.executable)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
c:\Users\user\AppData\Local\Programs\Python\Python311\python.exe


In [11]:
import os
import requests
from bs4 import BeautifulSoup

In [12]:
# 1. 설정
BASE_URL = "https://defense.na.go.kr:444/cmmit/cmitMtgRcord/mtgRcord/mtgRcordList.do"
PARAMS = {
    'menuNo': '2000071',
    'pageIndex': '1',
    'pageUnit': '50'  # 한 페이지에 50개를 불러오도록 설정
}
SAVE_DIR = "defense_minutes"

# 저장할 폴더 생성
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

def download_minutes():
    print("데이터를 불러오는 중입니다...")
    
    # 2. 웹페이지 요청 (SSL 인증서 검증 건너뛰기 - .go.kr 사이트 특성 대응)
    response = requests.get(BASE_URL, params=PARAMS, verify=False)
    if response.status_code != 200:
        print("페이지 접속에 실패했습니다.")
        return

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 3. 테이블 내의 모든 행(tr) 찾기
    # 이미지의 id="mtgRcordList-dataset-data-table" 내부의 tr을 타겟팅합니다.
    rows = soup.select("#mtgRcordList-dataset-data-table tr")
    
    print(f"총 {len(rows)}개의 항목을 발견했습니다.")

    for i, row in enumerate(rows):
        # 4. 다운로드 링크가 있는 a 태그 찾기
        # 이미지 구조상 마지막 td 안의 a 태그에 링크가 있음
        link_tag = row.select_one("td a[href*='download/pdf']")
        
        if link_tag:
            download_url = link_tag['href']
            # 상대 경로일 경우 절대 경로로 변환
            if not download_url.startswith("http"):
                download_url = "https://defense.na.go.kr:444" + download_url
            
            # 파일명 설정 (행의 텍스트나 alt 속성을 활용 가능)
            # 여기서는 alt 속성에 있는 회의명을 파일명으로 사용합니다.
            img_tag = link_tag.find("img")
            file_name = img_tag['alt'].replace(" pdf 다운로드", "").strip() if img_tag else f"meeting_{i+1}"
            file_name = "".join(c for c in file_name if c.isalnum() or c in (' ', '_', '-')).rstrip() # 특수문자 제거
            full_path = os.path.join(SAVE_DIR, f"{file_name}.pdf")

            # 5. 파일 다운로드 실행
            try:
                print(f"[{i+1}/{len(rows)}] 다운로드 중: {file_name}")
                file_res = requests.get(download_url, verify=False)
                with open(full_path, 'wb') as f:
                    f.write(file_res.content)
            except Exception as e:
                print(f"실패: {file_name} (오류: {e})")

    print("\n모든 작업이 완료되었습니다.")

if __name__ == "__main__":
    # SSL 경고 메시지 무시 설정
    requests.packages.urllib3.disable_warnings()
    download_minutes()

데이터를 불러오는 중입니다...
총 50개의 항목을 발견했습니다.
[1/50] 다운로드 중: 제22대 제429회 8차 국방위원회
[2/50] 다운로드 중: 제22대 제429회 2차 국방위원회 법률안심사소위원회
[3/50] 다운로드 중: 제22대 제429회 1차 국방위원회 법률안심사소위원회
[4/50] 다운로드 중: 제22대 제429회 7차 국방위원회
[5/50] 다운로드 중: 제22대 제429회 3차 국방위원회 예산결산심사소위원회
[6/50] 다운로드 중: 제22대 제429회 2차 국방위원회 예산결산심사소위원회
[7/50] 다운로드 중: 제22대 제429회 1차 국방위원회 군복지개선소위원회
[8/50] 다운로드 중: 제22대 제429회 6차 국방위원회
[9/50] 다운로드 중: 제22대 제429회 5차 국방위원회
[10/50] 다운로드 중: 제22대 제429회 4차 국방위원회
[11/50] 다운로드 중: 제22대 제429회 3차 국방위원회
[12/50] 다운로드 중: 제22대 제429회 2차 국방위원회
[13/50] 다운로드 중: 제22대 제429회 1차 국방위원회
[14/50] 다운로드 중: 제22대 제429회 1차 국방위원회 예산결산심사소위원회
[15/50] 다운로드 중: 제22대 제428회 1차 국방위원회 예산결산심사소위원회
[16/50] 다운로드 중: 제22대 제428회 1차 국방위원회
[17/50] 다운로드 중: 제22대 제427회 2차 국방위원회
[18/50] 다운로드 중: 제22대 제427회 1차 국방위원회
[19/50] 다운로드 중: 제22대 제426회 2차 국방위원회
[20/50] 다운로드 중: 제22대 제426회 1차 국방위원회 예산결산심사소위원회
[21/50] 다운로드 중: 제22대 제426회 1차 국방위원회
[22/50] 다운로드 중: 제22대 제424회 1차 국방위원회 법률안심사소위원회
[23/50] 다운로드 중: 제22대 제422회 3차 국방위원회
[24/50] 다운로드 중: 제22대 제422회 2차 국방위원회 법률안심사소위원회
[25